In [1]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, Subset
from dataset import IMDBDataset
from model import AgeClassifier
from torch.utils.data import random_split, SubsetRandomSampler
from torch import optim, nn
from tqdm import tqdm
from unlearn import *
from utils import *
from sklearn.model_selection import train_test_split
import random

In [ ]:
teacher_checkpoint_path = 'IMDB_CROP_Pretrained_Teacher.pt'
forget_checkpoint_path = 'IMDB_CROP_Pretrained_Forget.pt'

### Unzip Dataset

In [2]:
import tarfile
import os

if not os.path.isdir('./imdb_crop'):
    with tarfile.open('./imdb_crop.tar', 'r') as tar:
        tar.extractall('./')

In [3]:
dataset = IMDBDataset('./imdb_crop/imdb.mat', './imdb_crop', limit=4000)

train_idx, valid_idx = train_test_split(list(range(len(dataset))), test_size=0.2, random_state=42)

train_ds = Subset(dataset, train_idx)
valid_ds = Subset(dataset, valid_idx)

In [4]:
celeb_ids = torch.tensor(dataset.celeb_ids)

forget_celeb_id = random.choice(torch.unique(celeb_ids).tolist())

retain_train_ds = Subset(dataset, torch.tensor([i for i in train_idx if celeb_ids[i] != forget_celeb_id]))
retain_valid_ds = Subset(dataset, torch.tensor([i for i in valid_idx if celeb_ids[i] != forget_celeb_id]))

forget_train_ds = Subset(dataset, torch.tensor([i for i in train_idx if celeb_ids[i] == forget_celeb_id]))
forget_valid_ds = Subset(dataset, torch.tensor([i for i in valid_idx if celeb_ids[i] == forget_celeb_id]))

In [ ]:
device = 'cuda'

batch_size = 256
num_workers = 4

train_dl = DataLoader(train_ds, batch_size, num_workers=num_workers, pin_memory=False, shuffle=True)
valid_dl = DataLoader(valid_ds, batch_size, num_workers=num_workers, pin_memory=False)

retain_train_dl = DataLoader(retain_train_ds, batch_size, num_workers=num_workers, pin_memory=False, shuffle = True)
retain_valid_dl = DataLoader(retain_valid_ds, batch_size, num_workers=num_workers, pin_memory=False)

forget_train_dl = DataLoader(forget_train_ds, batch_size, num_workers=num_workers, pin_memory=False, shuffle = True)
forget_valid_dl = DataLoader(forget_valid_ds, batch_size, num_workers=num_workers, pin_memory=False)

In [ ]:

full_trained_teacher = AgeClassifier(num_classes = 5, pretrained = True).to(device)

# Training
history = fit_one_cycle(5, full_trained_teacher, train_dl, valid_dl, device = device)

# Loading
# full_trained_teacher.load_state_dict(torch.load("ResNET18_CIFAR100Super20_Pretrained_ALL_CLASSES_5_Epochs.pt", map_location = device))

# Saving
torch.save(full_trained_teacher.state_dict(), teacher_checkpoint_path)

C:\Users\Foopy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Foopy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\Foopy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torch\optim\lr_scheduler.py:62: UserWarn

Epoch [0], last_lr: 0.01000, train_loss: 1.7379, val_loss: 1.4931, val_acc: 23.3750
Epoch [1], last_lr: 0.01000, train_loss: 1.5942, val_loss: 12.9621, val_acc: 37.7500
Epoch [2], last_lr: 0.01000, train_loss: 1.5673, val_loss: 1.3079, val_acc: 38.6250
Epoch [3], last_lr: 0.01000, train_loss: 1.4653, val_loss: 1.5942, val_acc: 38.1250
Epoch [4], last_lr: 0.01000, train_loss: 1.4242, val_loss: 1.7141, val_acc: 38.5000


In [7]:
evaluate(full_trained_teacher, retain_valid_dl, device)

{'Loss': 1.7205034494400024, 'Acc': 37.900001525878906}

In [8]:
evaluate(full_trained_teacher, forget_valid_dl, device)

{'Loss': 0.9652605652809143, 'Acc': 75.0}

### Forget

In [ ]:
model = AgeClassifier(num_classes = 5, pretrained = False).to(device)
unlearning_teacher = AgeClassifier(num_classes = 5, pretrained = False).to(device)

# Training
model.load_state_dict(torch.load(teacher_checkpoint_path, map_location = device))
blindspot_unlearner(model = model, unlearning_teacher = unlearning_teacher, full_trained_teacher = full_trained_teacher, 
                    retain_data = retain_train_ds, forget_data = forget_train_ds, epochs = 1, lr = 0.0001, 
                    batch_size = batch_size, num_workers = num_workers, device = device)

# Loading
# model.load_state_dict(torch.load("ResNET18_CIFAR100Super20_Pretrained_Forget_Class69_1_Epochs.pt", map_location = device))

# Saving
torch.save(model.state_dict(), forget_checkpoint_path)

C:\Users\Foopy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Foopy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 1 Unlearning Loss 0.07934194803237915


In [11]:
evaluate(model, retain_valid_dl, device)

{'Loss': 2.1677489280700684, 'Acc': 37.900001525878906}

In [12]:
evaluate(model, forget_valid_dl, device)

{'Loss': 0.9760209918022156, 'Acc': 75.0}